# 08 · การทดสอบสมมติฐาน

รายวิชากำหนดให้ทดสอบสมมติฐานครบทุกรูปแบบประชากร แต่ละการทดสอบพิมพ์ 5 ขั้นตอน คือ
สมมติฐาน ระดับนัยสำคัญ ตัวสถิติทดสอบ ค่าสถิติกับค่า p และการตัดสินใจ

| รูปแบบ | คำถาม | การทดสอบ |
|---|---|---|
| 1 ประชากร ค่าเฉลี่ย | ชั่วโมงนอนเฉลี่ยต่างจาก 7 ชั่วโมงที่แนะนำหรือไม่ | one-sample t-test |
| 1 ประชากร สัดส่วน | สัดส่วนผู้ที่คะแนนความวิตกกังวลเกินเกณฑ์ปกติต่างจากครึ่งหนึ่งหรือไม่ | one-proportion z-test |
| 2 ประชากร ค่าเฉลี่ย | คะแนนภาวะซึมเศร้าของชายและหญิงต่างกันหรือไม่ | independent-samples t-test |
| k ประชากร ค่าเฉลี่ย | คะแนนความวิตกกังวลของ 4 กลุ่มเนื้อหาต่างกันหรือไม่ | one-way ANOVA |

ทุกการทดสอบใช้ระดับนัยสำคัญ 0.05 และมีการทดสอบแบบไม่อิงพารามิเตอร์ตรวจซ้ำในบรรทัด note

In [1]:
import os
import pathlib
import sys

# notebook.nvim starts the kernel in nvim's cwd; step into codingpy/ so common.py
# and data/ are found when the notebook is opened from the repo root
if pathlib.Path("codingpy/common.py").is_file():
    os.chdir("codingpy")
    sys.path.insert(0, os.getcwd())

%matplotlib inline

In [2]:
import numpy as np
import pandas as pd
from scipy import stats

from common import ALPHA, cohens_d, eta_squared, load_data, passed_attention

df = passed_attention(load_data())  # the sample every research question uses
print(f"{len(df)} respondents passed the attention check")

107 respondents passed the attention check


In [3]:
# DASS-21 upper bound of the "normal" band, per subscale (Lovibond & Lovibond)
NORMAL_CUTOFF = {"dass_depression": 9, "dass_anxiety": 7, "dass_stress": 14}

SLEEP_REFERENCE = 7.0  # hours: recommended minimum for adults


def steps(title, h0, h1, test, statistic, pvalue, decision, note=""):
    print(f"\n{title}")
    print(f"  H0        : {h0}")
    print(f"  H1        : {h1}")
    print(f"  alpha     : {ALPHA}")
    print(f"  test      : {test}")
    print(f"  statistic : {statistic}")
    print(f"  p-value   : {pvalue:.6g}")
    verdict = "reject H0" if pvalue < ALPHA else "fail to reject H0"
    print(f"  decision  : {verdict} -> {decision}")
    if note:
        print(f"  note      : {note}")


def one_prop_z(successes: int, n: int, p0: float):
    """z statistic and two-sided p for a single proportion."""
    phat = successes / n
    z = (phat - p0) / np.sqrt(p0 * (1 - p0) / n)
    return phat, z, 2 * (1 - stats.norm.cdf(abs(z)))


def one_population_mean(df: pd.DataFrame) -> None:
    x = df["sleep_hours"].dropna()
    t, p = stats.ttest_1samp(x, SLEEP_REFERENCE)
    steps(
        "1 population, mean: average sleep vs the 7-hour recommendation",
        f"mu = {SLEEP_REFERENCE} hours",
        f"mu != {SLEEP_REFERENCE} hours (two-sided)",
        f"one-sample t-test, df = {len(x) - 1} "
        f"(n = {len(x)}, mean = {x.mean():.2f}, SD = {x.std(ddof=1):.2f})",
        f"t = {t:.3f}",
        p,
        f"mean sleep {x.mean():.2f} h differs from {SLEEP_REFERENCE} h",
        f"Shapiro p = {stats.shapiro(x).pvalue:.2g}, but n = {len(x)} so the CLT covers "
        f"the t-test; Wilcoxon signed-rank agrees (p = {stats.wilcoxon(x - SLEEP_REFERENCE).pvalue:.2g})",
    )


def one_population_proportion(df: pd.DataFrame, col="dass_anxiety") -> None:
    above = (df[col] > NORMAL_CUTOFF[col]).sum()
    n = df[col].notna().sum()
    phat, z, p = one_prop_z(above, n, 0.5)
    steps(
        "1 population, proportion: share above the DASS-21 anxiety cut-off vs one half",
        "p = 0.50",
        "p != 0.50 (two-sided)",
        f"one-proportion z-test (x = {above}, n = {n}, p-hat = {phat:.3f}); "
        f"n*p0 = {n * 0.5:.0f} >= 10, so the normal approximation holds",
        f"z = {z:.3f}",
        p,
        f"{phat:.1%} score above the normal band, not half",
    )


def two_population_mean(df: pd.DataFrame, col="dass_depression") -> None:
    male = df.loc[df["gender"] == "Male", col].dropna()
    female = df.loc[df["gender"] == "Female", col].dropna()
    levene = stats.levene(male, female)
    equal_var = bool(levene.pvalue >= ALPHA)
    t, p = stats.ttest_ind(male, female, equal_var=equal_var)
    mw = stats.mannwhitneyu(male, female)
    kind = "pooled-variance" if equal_var else "Welch"
    steps(
        "2 populations, mean: depression score, male vs female",
        "mu_male = mu_female",
        "mu_male != mu_female (two-sided)",
        f"independent-samples t-test, {kind} "
        f"(Levene p = {levene.pvalue:.3f}); "
        f"male n = {len(male)}, mean = {male.mean():.2f}; "
        f"female n = {len(female)}, mean = {female.mean():.2f}",
        f"t = {t:.3f}, Cohen's d = {cohens_d(male, female):.2f}",
        p,
        f"male depression mean is {male.mean() - female.mean():.2f} points higher",
        f"Mann-Whitney U agrees (p = {mw.pvalue:.4f}); the Bonferroni line across "
        f"the three subscales is alpha = {ALPHA / 3:.4f}",
    )


def k_population_mean(df: pd.DataFrame, col="dass_anxiety") -> None:
    groups = [g[col].dropna().to_numpy(dtype=float) for _, g in df.groupby("content_group")]
    names = sorted(df["content_group"].dropna().unique())
    levene = stats.levene(*groups)
    F, p = stats.f_oneway(*groups)
    kw = stats.kruskal(*groups)
    k, n = len(groups), sum(len(g) for g in groups)
    sizes = ", ".join(f"{nm} n = {len(g)}, mean = {g.mean():.2f}" for nm, g in zip(names, groups))
    steps(
        "k populations, mean: anxiety score across the four content groups",
        "mu_News = mu_Entertainment = mu_OthersLife = mu_Other",
        "at least one group mean differs",
        f"one-way ANOVA, df = ({k - 1}, {n - k}); equal variances hold "
        f"(Levene p = {levene.pvalue:.3f}). {sizes}",
        f"F = {F:.3f}, eta^2 = {eta_squared(groups):.4f}",
        p,
        "no detectable difference in anxiety between content groups",
        f"Kruskal-Wallis agrees (p = {kw.pvalue:.4f}); the smallest group has only "
        f"{min(len(g) for g in groups)} respondents, so power against a small difference is low",
    )


one_population_mean(df)
one_population_proportion(df)
two_population_mean(df)
k_population_mean(df)


1 population, mean: average sleep vs the 7-hour recommendation
  H0        : mu = 7.0 hours
  H1        : mu != 7.0 hours (two-sided)
  alpha     : 0.05
  test      : one-sample t-test, df = 105 (n = 106, mean = 6.21, SD = 1.54)
  statistic : t = -5.244
  p-value   : 8.20824e-07
  decision  : reject H0 -> mean sleep 6.21 h differs from 7.0 h
  note      : Shapiro p = 0.0022, but n = 106 so the CLT covers the t-test; Wilcoxon signed-rank agrees (p = 1.4e-06)

1 population, proportion: share above the DASS-21 anxiety cut-off vs one half
  H0        : p = 0.50
  H1        : p != 0.50 (two-sided)
  alpha     : 0.05
  test      : one-proportion z-test (x = 73, n = 107, p-hat = 0.682); n*p0 = 54 >= 10, so the normal approximation holds
  statistic : z = 3.770
  p-value   : 0.00016307
  decision  : reject H0 -> 68.2% score above the normal band, not half

2 populations, mean: depression score, male vs female
  H0        : mu_male = mu_female
  H1        : mu_male != mu_female (two-sided)
  a

**อ่านผล**

- **ชั่วโมงนอน** เฉลี่ย 6.21 ชั่วโมง ต่ำกว่า 7 ชั่วโมงที่แนะนำอย่างมีนัยสำคัญ (t = −5.24, p < 0.001)
- **ความวิตกกังวล** ผู้ตอบ 68.2% (73 จาก 107 คน) มีคะแนนเกินเกณฑ์ปกติ มากกว่าครึ่งหนึ่งอย่างมีนัยสำคัญ (z = 3.77, p < 0.001)
- **เพศ** คะแนนภาวะซึมเศร้าของชายและหญิงไม่ต่างกัน (t = 1.05, p = 0.297, d = 0.20)
- **กลุ่มเนื้อหา** คะแนนความวิตกกังวลของ 4 กลุ่มไม่ต่างกัน (F = 0.23, p = 0.879) ตรงกับผล Kruskal-Wallis ใน `04_q4_content_type.ipynb`